# Incremental Ingestion (`ingest_nbk`)

Inject **new chunks + their precomputed embeddings** (already processed upstream, e.g. on HPC)
into the `optimized_*` family, then refresh the Vector Search index.

The upsert is idempotent (MERGE on `chunk_id`), so re-running with overlapping ids updates rows
in place rather than duplicating them.


In [0]:
# --- Make scripts/ importable ---
# In Databricks Repos the repo root is on sys.path; locate the scripts/ subdir there
# (works the same when running from the repo root locally).
import os, sys

_cands = [os.path.join(p, "scripts") for p in (list(sys.path) + [os.getcwd()])]
for _cand in _cands:
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import settings
from ingest import load_new, ingest, sync_vector_index, rebuild_bm25
from lake_io import read_chunks_jsonl, read_embeddings_npy

## 1. Point at the new (HPC) output

The new data is expected in the same layout as the lake:
```
{NEW_BASE}/optimized_chunks/*/chunks.jsonl
{NEW_BASE}/optimized_embeddings/{model}/*/embeddings.npy
{NEW_BASE}/optimized_embeddings/{model}/*/chunk_ids.json
```


In [0]:
# Location of the newly processed batch (edit per run)
NEW_BASE = "abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files_incoming/"

chunks_df, emb_df = load_new(spark, NEW_BASE)
print("new chunks    :", chunks_df.count())
print("new embeddings:", emb_df.count())
display(chunks_df.limit(5))

## 2. Ingest (upsert) + refresh BOTH retrieval paths

`ingest()` makes the new chunks reachable by hybrid search end-to-end:
- **keyword side** — rebuilds the BM25 tables (`update_keyword_index=True`, required; otherwise
  new chunks are invisible to keyword recall),
- **vector side** — syncs the Vector Search index (`sync_index=True`).

Your index `optimized_rag_chunks_vs` is a **Triggered** Delta Sync index on endpoint
`tdis-ai-rag-light` (preset in `settings.VS_ENDPOINT`), so the sync is required too — it does not
update on its own.

In [0]:
report = ingest(spark, chunks_df, emb_df, sync_index=True, wait_for_sync=True, update_keyword_index=True)
print(report)

## 3. Notes: incremental BM25 & batch pattern

BM25 is updated **incrementally** inside `ingest()` — only the chunks in this batch are
re-tokenized, and the four `optimized_kw_*` tables are patched in place (idempotent, handles
re-ingested chunk_ids). So you do **not** need a full rebuild per ingest.

- Full repair (e.g. after schema changes): `rebuild_bm25(spark)` recomputes everything from scratch.
- Many small batches: keep BM25 incremental per batch, but set `sync_index=False` and trigger the
  Vector Search sync **once** at the end (the Triggered sync scans the whole source table).

In [0]:
# Batch example: incremental BM25 each time, sync the index only once at the end
# for new_base in batch_dirs:
#     cdf, edf = load_new(spark, new_base)
#     ingest(spark, cdf, edf, sync_index=False, update_keyword_index=True)
#
# sync_vector_index(wait=True)
#
# # Full BM25 repair (rarely needed):
# # print(rebuild_bm25(spark))